<a href="https://colab.research.google.com/github/haze25102583/CNN/blob/main/day6_lesson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM·GRU와 텍스트 분류

# [SECTION 01] 텍스트 분류 문제, 입력 구조

### (code 1) Padding, truncation을 숫자로 확인

Truncation

        문장이 정해진 최대 길이보다 길 때,
        초과된 토큰을 잘라내는 것


pad

        배치 입력의 길이를 맞추기 위해 0으로 채우고,
        padding 0은 실제 단어가 아님 -> mask로 구분
        pad 위치:  LSTM·GRU의 상태 갱신에서 제외

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

texts = tf.constant([
    "good movie", "this movie is very good today"
])
vec = layers.TextVectorization(
    output_mode="int", output_sequence_length=4     # 길이 4로 맞춤.
)
vec.adapt(texts)        # Vocabulary 구축
print(vec(texts).numpy) # 0채움, 뒤쪽 절단 확인

<bound method _EagerTensorBase.numpy of <tf.Tensor: shape=(2, 4), dtype=int64, numpy=
array([[3, 2, 0, 0],
       [6, 2, 7, 4]])>>


### (code 2) Vocabulary·OOV·PAD 동작 한 번에 확인

TextVectorization의 Vocabulary
        
        훈련 데이터로만 만듦

vectorizer

        문자열을 정규화 -> token으로 나눈뒤,
        각 token을 Vocabulary의 정수ID로변환하는 전처리계층

In [ ]:
from tensorflow.keras import layers
texts = ["good movie", "bad film", "not good"]
vec = layers.TextVectorization(
    max_tokens=8, output_sequence_length=4)
vec.adapt(texts)     # Vocabulary 생성

vocab = vec.get_vocabulary()
ids = vec(["good story"])
print("vocab: ", vocab)             # vocab:  ['', '[UNK]', np.str_('good'), np.str_('not'), np.str_('movie'), np.str_('film'), np.str_('bad')]
print("ids  : ", ids.numpy())       # ids  :  [[2 1 0 0]]

assert vocab[:2] == ["", "[UNK]"]   # [UNK]: 미등록 토큰
assert ids.shape == (1, 4)
assert ids.numpy()[0, -1] == 0

vocab:  ['', '[UNK]', np.str_('good'), np.str_('not'), np.str_('movie'), np.str_('film'), np.str_('bad')]
ids  :  [[2 1 0 0]]


# [SECTION 02] Embedding과 masking

### (code 3) embedding의 출력 shape -> 3차원으로 읽기


token index를 학습 가능한 vector로 바꾸고,

padding을 제외한 실제 문맥만 순환층에 전달.

Embedding

        token ID에 해당하는 행 1을 조회.
        벡터는 학습을 통해 분류에 유용한 특성을 담도록 (= 분류 loss를 줄이는 방향) 업데이트


Embedding table

        층을 만들 때 생성.
        
        그 안의 벡터는 학습을 통해 형성
        Embedding vector: shape [D] -> D 개의 실수로 이루어진 벡터


Embedding 백터도 분류 모델과 함께 분류에 유용하도록 수정.

숫자의 사전적 의미x 긍정x 부정x

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
ids = tf.constant([             # N=2, T=3
    [2, 3, 0],
    [4, 2, 1]
])

embedding = layers.Embedding(                   # vocabulary 범위: 10,  vector 차원: 4
    input_dim=10, output_dim=4, mask_zero=True) # ID 0 -> PAD
vectors = embedding(ids)
print(ids.shape, "->", vectors.shape)           # (2, 3) -> (2, 3, 4)

(2, 3) -> (2, 3, 4)


### (code 4) mask_zero가 만든 True·False

layers.Embedding(mask_zero=True)

        Embedding은 0 위치를 지우지 않고,
        ID 0 -> PAD 위치로 표시.

        다음 순환층이 무시할 수 있는 mask로 표시.
        RNN(LSTM·GRU)은 PAD 위치를 문장 처리에 반영x

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
ids = tf.constant([
    [2, 3, 0, 0],
    [4, 2, 1, 0]
])
embedding = layers.Embedding(
    input_dim=10, output_dim=4, mask_zero=True)     # padding 0은 False

mask = embedding.compute_mask(ids)
print(mask.numpy())             # [[ True  True False False]
                                #  [ True  True  True False]]

[[ True  True False False]
 [ True  True  True False]]


Embedding 파라미터 수

        Param # = V x D
        (V: vocabulary 크기,     D: vector 차원의 크기)

        문장 길이 T는 토큰 위치의 수.
        단순히 출력 크기, 연산량, 메모리만 변환

model.summary()

        (배치크기, 토큰 자리 수, 토큰 하나를 표현하는 숫자 수)

        input_dim: 토큰 ID 0~input_dim 마다 (토큰 자리수)차원 벡터 한 행을 준비


shape가 다르면,

MAX_LEN, input_dim, output_dim을 확인

shape 흐름

1. 문장
2. 토큰 ID [B, 120]
3. Embedding [B, 120, 32]
4. 평균 Pooling [B, 32]
5. 확률 [B,1]

1D CNN(1차원 합성곱 신경망)

        커널이 토큰 순서 '한 방향'으로 이동.
        인접 토큰 조합 탐지.
        ex) 문장

2D CNN

        가로·세로방향으로 이동.
        이미지의 작은 영역 탐지.



Embedding -> 토큰 벡터: [N, T, D]

        1. Embedding 평균 +  Dense
            모든 벡터(T개)의 벡터를 평균 -> 문장 전체를 표현
            [N, T, D] -> [N, D]
            단어 순서 반영 x
            계산 빠름
        
        2. 1D CNN
            폭 k의 커널 -> 인접토큰을 함께 관찰
            층을 쌓으면, 관찰 범위가 넓어짐

        3. RNN 계열: SimpleRNNN·LSTM·GRU
            t=1부터 T까지, 순서대로(x병렬화x) 상태갱신
            LSTM·GRU: 장기 정보 보존

# [SECTION 03] 기본 RNN의 한계-> Gate설계

기본 RNN

        긴 시퀀스에서 앞쪽 정보 약해짐
        (모든 시점에서) 같은 RNN 셀, 가중치 -> 공유

        순전파: 과거 -> 미래
        BPTT: 미래 -> 과거로 기울기 전달


LSTM·GRU

        셀 상태(C_t)와 은닉 상태(h_t) 나누어 사용
        게이트:
        GRU: 구조 단순화 -> h_t

Gradient vanishing/ exploding
        
        역전파의 학습 문제
        시간축을 따라 gradient가 너무 작아지거나 커짐
        -> 초반 정보 학습 ↓
        -> BPTT, LSTM·GRU, gradient clippint

State overwrite

        순전파의 정보 보존 문제
        새로운 입력마다 입력 상태(h_t) 갱신
        ->  LSTM의 cell state·gate, GRU의 update gate


SimpleRNN

        Gate 없이 이전 상태 h_t-1과 현재 입력 x_t를 한번에 결합-> 갱신
        오래된 정보 해석


LSTM

        cell state (C_t), hidden state (hₜ) 함께 Gate로 조절
        각 Gate는 현재 입력 xₜ, 이전 hidden state hₜ₋₁-> 비율을 결정

        기억, 출력 분리
    

GRU

        hidden state (hₜ) 만 Gate로 조절
        별도의 cell state 없이, hₜ 하나를 저장-> 다음 시점으로 전달

Gate : 순전파 상태 제어

        0~1의 반영 비율
        상태 덮어쓰기(state overwrite), 기울기 소실을 완화
        Forget gate:  이전 기억을 얼마나 남길지
        input gate: 반영할 새 후보
    

cell state

        forget gate + input gate -> new cell state

차원
        
        백터 안의 한 칸
        원소별 곱 x 게이트 값 -> 0 근사: 크게 억제, 1: 많이 통과

        각 자리에 고정x, 현재와 이전 상태 따라 매 시점 새롭게 계산

        unit=n이면, 상태 벡터와 게이트 벡터가 각각 n개의 값을 가짐

RNN

        장기 의존성 : 멀리 떨어진 앞 단서가 뒤의 해석에 영향을 주는 관계
        최종 판단에 필요한 단서만 선택적으로 오래 유지

        초기 부정 단서 자체가 이동x, 영향이 은닉상태(h_t)에 담겨 전달
        새 토큰 마다 h_t 갱신
        문장이 길면 초기 단서의 영향이 희미

### (code 6) SimpleRNN·LSTM·GRU 비용

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
x = tf.zeros((2, 12, 4))            # (batch_size, timestep, feature)

models = [layers.SimpleRNN(8),      # unit=8
          layers.LSTM(8),
          layers.GRU(8)]

params=[]

for layer in models:
    y = layer(x)
    params.append(layer.count_params())
    print(layer.__class__.__name__, y.shape,    # SimpleRNN (2, 8) 104
          layer.count_params())                 # LSTM (2, 8) 416
                                                # GRU (2, 8) 336
assert params ==[104, 416, 336]

# 같은 출력 크기지만,
# 내부 gate와 bias 묶음 때문에 학습 파라미터 수가 다름

SimpleRNN (2, 8) 104
LSTM (2, 8) 416
GRU (2, 8) 336


# [SECTION 04] LSTM과 GRU 구조 비교

LSTM

        기억통로, 현재출력 요약 -> 분리
        중요 정보를 덜 덮어씀

Cₜ

        장기 기억해야 할 정보
        forget·input으로 갱신

hₜ

        현재출력요약
        output gate를 거침 -> 다음 층, timestep에 전달
        마지막 hT: Dense sigmoid -> 분류

gate

        0~1 vector
        각 차원의 통과량




LSTM의 한 timestep

        ------  < 기억 제어> -------
        1. 입력결합: concat(xₜ, hₜ₋₁)
        2. Forget gate: 이전C 유지량
        3. Input gate + 새후보

        ------ < 장기기억 갱신> ------
        4. Cell update -> Cₜ 갱신
        
        ------ < 현재 출력 생성> ------
        5. Output gate : 꺼낼 정보 선택
        6. Hidden: hₜ 생성




Input gate

        현재입력, 이전 상태 정보 -> 반영할 정보 선택 -> 현재 기억 완성

Output gate

        Cₜ에서 hₜ로 내보낼 양을 조절

Cell state (Cₜ)

        Forget·Input gate 거쳐 갱신된 '장기 기억'

Output gate (oₜ)

        xₜ, hₜ₋₁-> 차원별 통과 비율
        oₜ: 0~1 사이의 벡터

Hidden state (hₜ)

        hₜ = oₜ ⊙ tanh(Cₜ)
        (⊙: 같은위치의 값끼리 곱)
        (tanh(Cₜ): 기억을 -1~1 범위의 출력후보로 바꿈)
    
        현재 출력 = 다음timestep으로 넘길 요약

분류 연결

        1. 마지막 hₜ
        2. Dropout
        3. Dense(1, sigmoid)
        4. 긍정 확률

### (code 7) LSTM의 output·h·C shape

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

x = tf.zeros((2,4,3))

# unit=5인 LSTM의 마지막 상태를 함께 받음
output, h, c = layers.LSTM(
    5, return_state=True)(x)
print("output: ", output.shape)         # output:  (2, 5)
print("h     : ", h.shape)              # h     :  (2, 5)
print("c     : ", c.shape)              # c     :  (2, 5)
print("output==h: ", tf.reduce_all(output==h).numpy())  # output==h:  True

# 마지막 output == h
# C는 별도의 장기 기억

output:  (2, 5)
h     :  (2, 5)
c     :  (2, 5)
output==h:  True


GRU

        두 gate -> hidden state 갱신

RESET GATE

        새 후보를 만들때, 과거 상태를 얼마나 참고할 지

UPDATE GATE

        이전 상태, 새 후보 -> 어떤 비율로 섞을 지

HIDDEN STATE

        cell state 없이, hidden state 하나만 갱신




```
# MODEL CODE

inputs = keras.Input(shape=(), dtype="string")
token_ids = vectorizer(inputs)
x = layers.Embedding(
    8000, 32, mask_zero=True)(token_ids)
x = layers.GRU(32)(x)
output = layers.Dense(
    1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)```



1. Reset gate

        후보 상태를 위한 과거 정보 -> 조절

2. Candidate state

        현재 입력, 조절된 이전 상태 -> 새로운 정보의 후보 생성

3. Update gate

        최종 상태에 남길 정보를 선택


4. Final hidden

        두 상태를 혼합 -> 다음 시점으로 전달할 최종 상태
        기억전달과 출력




gate 값: 입력마다 계산 <- 가중치 (loss를 통해 학습)

GRU의 새 hidden state

    1. 새 후보 만들기

        Reset: 후보를 만들기 전에 과거 참고량을 정함


    2. 최종 상태 만들기

        Update gate: 최종 상태에서 과거, 후보의 혼합 비율을 정함

### (code 8) GRU는 마지막 상태 하나만 반환

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

x = tf.zeros((2, 4, 3))

output, h = layers.GRU(5, return_state=True)(x) # GRU: cell state (C)를 반환x

print("output: ", output.shape)     # output:  (2, 5)
print("h     : ", h.shape)          # h     :  (2, 5)
print("output==h: ", tf.reduce_all(output==h).numpy())     # output==h:  True

output:  (2, 5)
h     :  (2, 5)
output==h:  True


LSTM, GRU의 상태 수, gate 구성

validation 성능·속도·파라미터·오류 유형으로 선택

LSTM

    상태    : cell (C) + hidden (h)
    Gate    :  forget·input·output
    표현력  : 기억, 출력 -> 분리
    파라미터: 같은 units에서 더 많음

GRU

    상태    :  hidden h 중심
    Gate    :  reset·update
    표현력  :  단순, 효율적 갱신
    파라미터: 같은 units에서 더 적음


### (code 9) LSTM, GRU 파라미터 수

학습 파라미터 수

게이트, 후보 상태 계산 ↑ -> 학습 파라미터 ↑

SimpleRNN

        U(D+U+1)

LSTM

        4U(D+U+1)

GRU

        3U(D+U+2)
        (+2 : input bias + recurrent bias)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
x = tf.zeros((1, 5, 4))
lstm = layers.LSTM(8)
gru = layers.GRU(8)

# 첫 호출로 가중치를 만듦 -> 개수 count
lstm(x)
gru(x)

print("LSTM: ", lstm.count_params())        # 416
print("GRU : ", gru.count_params())         # 336
# 같은 unit 이지만 gate 구조 때문에 비용이 다름

LSTM:  416
GRU :  336


# [SECTION 05] IMDB 감성 분류

여섯 단계를 하나의 Keras 모델

1. 문장
2. token
3. index
4. vector
5. 문맥
6. 확률

새 문자열을 예측

모델의 입력 방법

    val_loss, test 지표, 시간, 오류 문장 -> 선택
    마지막 정확도 << ** validation loss 최소 시점 **

평균 Embedding

    GlobalAveragePooling1D
    약한 순서 정보
    가장 빠른 기준선
    가장 적은 Param

LSTM

    h, c를 분리
    긴 단서 보존 후보
    가장 많은 Param

GRU

    h 중심 gated
    효율적 비교 후보
    중간 개수의 Param

혼동 행렬

    Actual 0(부정), 1(긍정)

    1. Pred0(부정)
        TN 420, FN 55
        FN -> 부정을 긍정으로
        문장 예시:  not perfect, but worth watching

    2. Pred1(긍정)
        FP 80, TP 445
        FP -> 부정을 긍정으로
        문장 예시: great acting, but painfully slow